# Who Needs tanh When You Have x³?:🤣

A totally normal experiment where we replace a well-behaved activation function
with a polynomial that *really* wants to explode — and then spend the rest of the
notebook trying to stop it from doing that.

**Spoiler:** normalization + mild clipping gets you to 23/30. That's fine. We're fine.

| Thing | Value |
|-------|-------|
| Activation | `x³` (chaotic neutral) |
| Network | MLP 3 → [4, 4, 1] |
| Dataset | 3 whole samples (yes, three) |
| Loss | MSE |
| Vibes | Anxious but optimistic |


## 1 · The Autograd Engine That Powers This Madness

Hand-rolled reverse-mode AD. Every scalar gets wrapped in a `Value`,
every operation builds a graph, and `backward()` walks it in reverse to accumulate gradients.

No PyTorch was harmed in the making of this notebook.


In [ ]:
import random
import math


class Value:
    """
    A scalar value that knows its own gradient.

    Wraps a float and tracks how it was computed so we can
    run backprop on it later. Think of it as a very anxious number.
    """

    def __init__(self, data, child=()):
        self.data = data
        self.grad = 0.0          # starts clueless, gets educated by backward()
        self._prev = set(child)
        self._backward = lambda: None  # no-op until an op overrides it

    def __repr__(self):
        return f"data = {self.data}"

    def __add__(self, other):
        """Addition. Gradient splits equally — democracy in action."""
        other = other if isinstance(other, Value) else Value(other)
        out = Value((self.data + other.data), (self, other))
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        """Subtraction. One side gets the gradient, the other gets the negative."""
        other = other if isinstance(other, Value) else Value(other)
        out = Value((self.data - other.data), (self, other))
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += -1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        """Multiplication. Each input's gradient is the *other* input's value. Classic swap."""
        other = other if isinstance(other, Value) else Value(other)
        out = Value((self.data * other.data), (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        """Power rule. Only works with plain numbers as exponents — no Value^Value nonsense."""
        assert isinstance(other, (int, float)), "only use int/float to power"
        out = Value((self.data ** other), (self,))
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def x_3(self):
        """
        The star of the show: f(x) = x³, f'(x) = 3x².

        Perfectly well-behaved near zero. Absolutely feral for large inputs.
        You've been warned.
        """
        return self ** 3

    def __neg__(self):
        """Unary minus. Flips the sign, flips the gradient."""
        return self * -1

    def __rmul__(self, other):
        """Allows `2 * value` in addition to `value * 2`."""
        return self * other

    def __truediv__(self, other):
        """Division via multiplication by the inverse. Works until other ≈ 0."""
        return self * (other ** -1)

    def backward(self):
        """
        Run backpropagation from this node.

        Builds a topological ordering of the graph, sets this node's
        gradient to 1.0, then calls each node's _backward in reverse order.
        """
        visited = set()
        topo = []
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


## 2 · The Network (Three Layers of Cubic Chaos)

Standard MLP structure — nothing fancy, except every hidden neuron fires through `x³`
instead of something sensible like `tanh`.

The output neuron is **linear** (no activation), so at least *something* in here
is behaving itself.


In [ ]:
class Neuron:
    """
    One neuron: computes w·x + b, then optionally applies x³.

    Args:
        nin: number of input features
        nonlin: if True, apply x³ activation. If False, stay linear (output layer).
    """

    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x):
        """Forward pass: dot product → bias → activation (if nonlin)."""
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.x_3() if self.nonlin else act

    def __repr__(self):
        return f"Neuron(w={self.w})"

    def parameters(self):
        """Returns all trainable parameters: weights + bias."""
        return self.w + [self.b]


class Layer:
    """
    A row of neurons that all receive the same input and fire in parallel.

    Args:
        nin: input size (same for all neurons in the layer)
        nout: how many neurons (= output size of this layer)
    """

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        """Run every neuron on x, return their outputs as a list."""
        return [n(x) for n in self.neurons]

    def __repr__(self):
        return f"Layer = {self.neurons}"

    def parameters(self):
        """Flatten all neuron parameters into one list."""
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    """
    Multi-Layer Perceptron: chains layers so each layer feeds the next.

    Args:
        nin: input feature count
        nouts: list of layer sizes, e.g. [4, 4, 1]

    The last layer is always linear (no x³) because we have some dignity.
    """

    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.Layers = [Layer(a, b) for a, b in zip(sizes, sizes[1:])]

    def __call__(self, x):
        """Forward pass: feed x through every layer in sequence."""
        for layer in self.Layers:
            x = layer(x)
        return x

    def __repr__(self):
        return f"MLP(Layers={self.Layers})"

    def parameters(self):
        """All parameters across all layers. Used by the optimizer loop."""
        return [p for layer in self.Layers for p in layer.parameters()]


## 3 · The Dataset (All Three of Them)

Yes, three samples. This is a proof-of-concept, not ImageNet.

The raw feature values range from 1 to 97. Run them through `x³` unscaled
and you get numbers like **912,673**. That's not a loss — that's a cry for help.

**Fix:** divide each sample's features by the sample's max absolute value → everything lands in `[-1, 1]`.


In [ ]:
xs = [[19, 1, 38], [55, 16, 60], [57, 15, 97]]
ys = [1, 0, 1]


def normalize(x):
    """
    Per-sample max-abs normalization.

    Divides all features by the largest absolute value in that sample,
    so the output range is [-1, 1]. Fast, simple, good enough for x³ to not implode.
    """
    max_value = max(abs(v) for v in x)
    return [(v / max_value) for v in x]


xs_norm = [normalize(x) for x in xs]

print("Raw inputs:        ", xs)
print("Normalized inputs: ", [list(map(lambda v: round(v, 3), x)) for x in xs_norm])
print()
mlp = MLP(3, [4, 4, 1])
print(f"Model: {mlp}")
print(f"Total parameters: {len(mlp.parameters())}  (not a lot, but they try hard)")


## 4 · Strategy 1 — Just Normalize It (Baseline)

**Setup:** normalized inputs · `lr = 0.001` · no clipping · 5 000 steps

The bare minimum to keep x³ from going off the rails.
It works — slowly. The loss crawls downward like it's hiking uphill in wet socks.


In [ ]:
import matplotlib.pyplot as plt

losses = []

for step in range(5000):
    # forward pass
    ypred = [mlp(x)[0] for x in xs_norm]
    loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))

    # backward pass
    loss.backward()

    # vanilla SGD — no tricks
    for p in mlp.parameters():
        p.data -= 0.001 * p.grad
    for p in mlp.parameters():
        p.grad = 0.0   # zero grad manually (no .zero_grad() here, we're in the wilderness)

    losses.append(loss.data)

print(f"Targets:     {ys}")
print(f"Predictions: {[round(p.data, 4) for p in ypred]}")
print(f"Final loss:  {losses[-1]:.6f}  (started at {losses[0]:.6f})")

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title("Strategy 1 — Normalization only (lr=0.001, no clipping)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.yticks([max(losses), min(losses)])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Uncomment to log to W&B:
# import wandb
# wandb.init(project="x3-activation-experiment", name="strategy1")
# wandb.log({"loss": loss.data, "step": step, "loss_delta": loss.data - losses[0]})


## 5 · Strategy 2 — Brute-Force Clipping on Raw Inputs (RIP)

**Setup:** raw inputs (no normalization) · `lr = 1e-7` · gradient clipped to `±1e-6` · 30 trials

The idea: *"what if I just clamp the gradients so hard that nothing can explode?"*

**Result: 0 / 30 converged. Every single trial exploded. 🔥**

**Why:** `x³(97) ≈ 912,673`. Even clipped to ±1e-6 and with a microscopic lr,
the pre-activation values are so astronomical that the network can't learn anything useful —
and numerical instability wins anyway. Normalization isn't optional with polynomial activations.


In [ ]:
results2 = []

for trial in range(30):
    mlp2 = MLP(3, [4, 4, 1])
    result = "unknown"

    for step in range(5000):
        ypred = [mlp2(x)[0] for x in xs]          # raw inputs — living dangerously
        loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))
        loss.backward()

        for p in mlp2.parameters():
            # clip gradients to ±1e-6 — more of a prayer than a technique
            p.grad = max(-0.000001, min(0.000001, p.grad))
            p.data -= 0.0000001 * p.grad
        for p in mlp2.parameters():
            p.grad = 0.0

        if math.isnan(loss.data) or loss.data > 1e6:
            result = "exploded"
            break
        if loss.data < 2.001:
            result = "converged"
            break

    results2.append(result)
    print(f"trial {trial:2d}: {result}")

print(f"\nStrategy 2 success rate: {results2.count('converged')}/30  (as expected)")


## 6 · Strategy 3 — Normalize + Clip (The One That Actually Works)

**Setup:** normalized inputs · `lr = 0.001` · gradient clipped to `±1.0` · 30 trials

Combine both techniques:
- Normalization keeps x³ from producing absurd outputs
- Mild clipping handles the occasional bad random initialization

**Result: 23 / 30 converged (~77%)**

The 7 failures are pure bad luck with weight initialization — x³'s derivative `3x²`
can spike hard if weights start too large. A proper init scheme (He-style, tuned for x³)
would probably fix most of those.


In [ ]:
results3 = []

for trial in range(30):
    mlp3 = MLP(3, [4, 4, 1])
    result = "unknown"

    for step in range(5000):
        ypred = [mlp3(x)[0] for x in xs_norm]     # normalized — we learned our lesson
        loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))
        loss.backward()

        for p in mlp3.parameters():
            p.grad = max(-1.0, min(1.0, p.grad))  # gentle guardrails, not a straitjacket
            p.data -= 0.001 * p.grad
        for p in mlp3.parameters():
            p.grad = 0.0

        if math.isnan(loss.data) or loss.data > 1e6:
            result = "exploded"
            break
        if loss.data < 2.001:
            result = "converged"
            break

    results3.append(result)
    print(f"trial {trial:2d}: {result}")

print(f"\nStrategy 3 success rate: {results3.count('converged')}/30  🎉")


## 7 · So What Did We Learn?

### The results, in a table, because tables look professional

| Strategy | Inputs | Clipping | Converged |
|----------|--------|----------|-----------|
| 1 — Normalize only | ✅ Normalized | ❌ None | ✅ Yes (slowly) |
| 2 — Clip only | ❌ Raw | 🔬 ±1e-6 | **0 / 30** 💀 |
| 3 — Normalize + Clip | ✅ Normalized | ✅ ±1.0 | **23 / 30** 🎉 |

### Honest takeaways

- **x³ is not tanh.** It doesn't saturate — it just keeps growing. Respect it.
- **Normalization is mandatory**, not optional, when your activation has no upper bound.
- **Gradient clipping is insurance** — it won't save you from bad inputs, but it
  handles bad weight init pretty well.
- **77% is genuinely okay** for a from-scratch engine with random init and no lr schedule.